In [48]:
from transformers import pipeline as hf_pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
import json
import re
from typing import Optional, List, Tuple
import torch
import numpy as np
import faiss
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
def load_model(model_name: str = "microsoft/Phi-3-mini-4k-instruct"):
    """
    Load a HuggingFace text-generation pipeline.

    We use Phi-3-mini because:
    - Small (3.8B params), runs on CPU or consumer GPU
    - Instruction-tuned (understands chat format)
    - Good reasoning for its size

    Alternatives if you have more VRAM:
    - "mistralai/Mistral-7B-Instruct-v0.2" (7B)
    - "meta-llama/Meta-Llama-3-8B-Instruct" (8B, requires HF login)
    """

    print(f"Loading model: {model_name}")
    print("This may take a few minutes on first run (downloading ~2GB)...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    pipe = hf_pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        torch_dtype=torch.float16,
        device_map="auto",  # auto-selects CPU/GPU
        max_new_tokens=256,
        do_sample=False,    # deterministic for tutorials
        temperature=1.0,    # irrelevant when do_sample=False
    )
    print("Model loaded successfully!\n")
    return pipe


def generate(pipe, messages: list[dict], max_new_tokens: int = 256) -> str:
    """
    Generate a response from a list of chat messages.

    Messages format:
        [{"role": "system", "content": "..."},
         {"role": "user",   "content": "..."}]

    Returns the assistant's response as a string.
    """
    output = pipe(messages, max_new_tokens=max_new_tokens)
    # Extract the generated text (remove the prompt from the output)
    response = output[0]["generated_text"]
    # The last message in the conversation is the assistant's response
    if isinstance(response, list):
        return response[-1]["content"]
    return response



## Zero shot prompting

In [3]:
def demo_zero_shot(pipe):
    """
    Zero-shot prompting: ask directly without any examples. Where, the model relies entirely on knowledge from pretraining.
    """
    print("=" * 60)
    print("SECTION 1: Zero-Shot Prompting")
    print("=" * 60)

    print("\n--- 1a. Basic Zero-Shot ---")
    messages = [
        {"role": "user", "content": "What is the band gap of silicon at room temperature?"}
    ]
    response = generate(pipe, messages)
    print(f"Prompt: {messages[0]['content']}")
    print(f"Response: {response}\n")

    print("--- 1b. Zero-Shot with Role Assignment ---")
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert materials scientist specializing in semiconductors. "
                "Be precise, cite numbers, and use SI units."
            )
        },
        {
            "role": "user",
            "content": "What is the band gap of silicon at room temperature?"
        }
    ]
    response = generate(pipe, messages)
    print(f"System: {messages[0]['content'][:60]}...")
    print(f"Prompt: {messages[1]['content']}")
    print(f"Response: {response}\n")


## Few shot prompting

In [4]:
def demo_few_shot(pipe):
    """
    Few-shot prompting: provide examples (demonstrations) before your query.
    """
    print("=" * 60)
    print("SECTION 2: Few-Shot Prompting")
    print("=" * 60)

    print("\n--- 2a. Few-Shot Classification ---")
    messages = [
        {
            "role": "system",
            "content": "Classify materials by their primary bonding type."
        },
        {
            "role": "user",
            "content": (
                "Examples:\n"
                "Material: NaCl -> Bonding: Ionic\n"
                "Material: Diamond -> Bonding: Covalent\n"
                "Material: Copper -> Bonding: Metallic\n"
                "Material: Water -> Bonding: Covalent\n\n"
                "Now classify:\n"
                "Material: Silicon -> Bonding:"
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=50)
    print(f"Response: {response}")

    print("\n--- 2b. Few-Shot Structured Extraction ---")
    messages = [
        {
            "role": "system",
            "content": "Extract material properties from text. Always respond in the exact format shown."
        },
        {
            "role": "user",
            "content": (
                "Examples:\n"
                'Text: "Gold melts at 1064°C and has density 19.3 g/cm³"\n'
                'Output: {"material": "Gold", "melting_point_C": 1064, "density_g_cm3": 19.3}\n\n'
                'Text: "Copper has electrical conductivity of 5.96e7 S/m and melts at 1085°C"\n'
                'Output: {"material": "Copper", "melting_point_C": 1085, "conductivity_S_m": 5.96e7}\n\n'
                'Now extract from:\n'
                'Text: "Aluminum has density 2.7 g/cm³ and melting point 660°C"\n'
                'Output:'
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=100)
    print(f"Response: {response}\n")

## Chain of thought prompting

In [5]:
def demo_chain_of_thought(pipe):
    """
    Chain-of-Thought (CoT): Forces the model to allocate more compute (tokens) to intermediate reasoning steps, similar to iterative refinement in optimization.
|   """
    print("=" * 60)
    print("SECTION 3: Chain-of-Thought Prompting")
    print("=" * 60)

    print("\n--- 3a. Direct Answer (Baseline) ---")
    messages = [
        {
            "role": "user",
            "content": (
                "A steel beam is at 20°C. The thermal expansion coefficient of steel "
                "is 12e-6 /°C. If the beam is 10 meters long and heated to 200°C, "
                "what is the new length?"
            )
        }
    ]
    response_direct = generate(pipe, messages, max_new_tokens=100)
    print(f"Direct response: {response_direct}")

    print("\n--- 3b. Chain-of-Thought ---")
    messages = [
        {
            "role": "system",
            "content": "Solve physics problems step by step. Show your work clearly."
        },
        {
            "role": "user",
            "content": (
                "A steel beam is at 20°C. The thermal expansion coefficient of steel "
                "is 12e-6 /°C. If the beam is 10 meters long and heated to 200°C, "
                "what is the new length?\n\n"
                "Let's think step by step:\n"
                "Step 1: Identify the formula for thermal expansion.\n"
                "Step 2: Identify given values.\n"
                "Step 3: Calculate the change in length.\n"
                "Step 4: Calculate the new length.\n"
                "Step 5: State the answer with units."
            )
        }
    ]
    response_cot = generate(pipe, messages, max_new_tokens=300)
    print(f"CoT response: {response_cot}\n")

    print("--- 3c. Zero-Shot CoT (Magic Phrase) ---")
    messages = [
        {
            "role": "user",
            "content": (
                "If we mix 60% iron, 20% chromium, and 20% nickel by weight, "
                "approximately what type of alloy do we get and what are its key properties? "
                "Think step by step."
            )
        }
    ]
    response_zs_cot = generate(pipe, messages, max_new_tokens=200)
    print(f"Zero-shot CoT response: {response_zs_cot}\n")



## Prompt Template

In [6]:
class PromptTemplate:
    """
    A reusable, parameterized prompt template.

    Similar to a function with default arguments — lets you standardize prompt structure while varying the inputs.
    """

    def __init__(self, template: str, input_variables: list[str]):
        self.template = template
        self.input_variables = input_variables

    def format(self, **kwargs) -> str:
        """Fill in the template with provided values."""
        missing = set(self.input_variables) - set(kwargs.keys())
        if missing:
            raise ValueError(f"Missing template variables: {missing}")
        return self.template.format(**kwargs)


def demo_prompt_templates(pipe):
    """
    Prompt templates standardize your prompts for reproducibility and batch processing — essential for building pipelines.
    """
    print("=" * 60)
    print("SECTION 4: Prompt Templates")
    print("=" * 60)

    material_property_template = PromptTemplate(
        template=(
            "You are a materials science database. Provide accurate, concise information.\n\n"
            "Material: {material}\n"
            "Property requested: {property}\n"
            "Units: {units}\n"
            "Answer (number and brief context only):"
        ),
        input_variables=["material", "property", "units"]
    )

    comparison_template = PromptTemplate(
        template=(
            "Compare {material_a} and {material_b} in terms of {criterion}. "
            "Format your response as:\n"
            "{material_a}: [value/description]\n"
            "{material_b}: [value/description]\n"
            "Verdict: [which is better for {use_case} and why in one sentence]"
        ),
        input_variables=["material_a", "material_b", "criterion", "use_case"]
    )

    queries = [
        {"material": "Silicon", "property": "band gap", "units": "eV"},
        {"material": "GaAs",    "property": "band gap", "units": "eV"},
        {"material": "Copper",  "property": "electrical resistivity", "units": "Ω·m"},
    ]

    print("\n--- Material Property Queries ---")
    for q in queries:
        prompt_text = material_property_template.format(**q)
        messages = [{"role": "user", "content": prompt_text}]
        response = generate(pipe, messages, max_new_tokens=80)
        print(f"{q['material']} - {q['property']}: {response.strip()[:100]}")

    # Use the comparison template
    print("\n--- Material Comparison ---")
    prompt_text = comparison_template.format(
        material_a="Steel",
        material_b="Aluminum",
        criterion="strength-to-weight ratio",
        use_case="aerospace structural components"
    )
    messages = [{"role": "user", "content": prompt_text}]
    response = generate(pipe, messages, max_new_tokens=150)
    print(f"Comparison result:\n{response}\n")

In [7]:
def demo_structured_output(pipe):
    """
    Force the model to generate structured outputs (JSON, YAML, etc.) by including format instructions and examples in the prompt. 
    """
    print("=" * 60)
    print("SECTION 5: Structured Output Generation")
    print("=" * 60)

    print("\n--- 5a. JSON Output ---")
    json_schema = {
        "material_name": "string",
        "crystal_structure": "string (e.g., FCC, BCC, HCP)",
        "melting_point_K": "number",
        "applications": ["string"],
        "is_semiconductor": "boolean"
    }

    messages = [
        {
            "role": "system",
            "content": (
                "You are a materials database API. Always respond with valid JSON only. "
                "No explanation, no markdown, just the JSON object."
            )
        },
        {
            "role": "user",
            "content": (
                f"Return a JSON object for Titanium with this exact schema:\n"
                f"{json.dumps(json_schema, indent=2)}\n\n"
                "Fill all fields with accurate values. Applications list should have 3 items."
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=200)
    print(f"Raw response: {response}")

    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            print(f"Parsed successfully: {json.dumps(parsed, indent=2)}")
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        print("Tip: Try adding 'Respond with ONLY the JSON, no other text' to the prompt")

    print("\n--- 5b. Structured List Output ---")
    messages = [
        {
            "role": "user",
            "content": (
                "List the top 5 metals by electrical conductivity. "
                "Format as a numbered list:\n"
                "1. [Metal] - [Conductivity in MS/m] - [Main use]\n"
                "2. ...\n\n"
                "Start your response with '1.' only."
            )
        }
    ]
    response = generate(pipe, messages, max_new_tokens=150)
    print(f"Response:\n{response}\n")

In [8]:

def main():
    print("\n" + "=" * 60)
    print("Foundations of Prompting")
    print("=" * 60 + "\n")

    # Load the model once (reuse for all sections)
    pipe = load_model()

    # Run all demonstrations
    demo_zero_shot(pipe)
    demo_few_shot(pipe)
    demo_chain_of_thought(pipe)
    demo_prompt_templates(pipe)
    demo_structured_output(pipe)

    print("\n" + "=" * 60)
    print("Tutorial complete!")
    print("=" * 60)


if __name__ == "__main__":
    main()


Foundations of Prompting

Loading model: microsoft/Phi-3-mini-4k-instruct
This may take a few minutes on first run (downloading ~2GB)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded successfully!

SECTION 1: Zero-Shot Prompting

--- 1a. Basic Zero-Shot ---


...

Copper - electrical resistivity: 1.68 x 10^-8 Ω·m

--- Material Comparison ---


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Comparison result:
 Steel: 1.5
Aluminum: 2.7
Verdict: Aluminum is better for aerospace structural components due to its higher strength-to-weight ratio, which is crucial for reducing overall aircraft weight while maintaining structural integrity.

SECTION 5: Structured Output Generation

--- 5a. JSON Output ---


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw response:  ```json

{

  "material_name": "Titanium",

  "crystal_structure": "Hexagonal Close-Packed (HCP)",

  "melting_point_K": 1941,

  "applications": [

    "Aerospace components",

    "Medical devices",

    "Chemical processing equipment"

  ],

  "is_semiconductor": false

}

```
Parsed successfully: {
  "material_name": "Titanium",
  "crystal_structure": "Hexagonal Close-Packed (HCP)",
  "melting_point_K": 1941,
  "applications": [
    "Aerospace components",
    "Medical devices",
    "Chemical processing equipment"
  ],
  "is_semiconductor": false
}

--- 5b. Structured List Output ---
Response:
 1. Silver - 63 MS/m - Used in high-quality conductors and contacts

2. Copper - 59.6 MS/m - Used in electrical wiring and electronics

3. Gold - 45.2 MS/m - Used in high-reliability connectors and corrosion-resistant applications

4. Aluminum - 37.7 MS/m - Used in power transmission lines and lightweight conductors

5. Platinum - 31.8 MS/m - Used in laboratory equipment and hi

### Making use if above techniques to extract useful info as materials engineer!

In [9]:
def load_pipeline():
    """Load the model"""
    pipe = hf_pipeline(
        "text-generation",
        model="microsoft/Phi-3-mini-4k-instruct",
        torch_dtype=torch.float16,
        device_map="auto",
        max_new_tokens=256,
        do_sample=False,
    )
    return pipe


def generate(pipe, messages, max_new_tokens=256):
    output = pipe(messages, max_new_tokens=max_new_tokens)
    response = output[0]["generated_text"]
    if isinstance(response, list):
        return response[-1]["content"]
    return response


### Pr (Prompting) Exercise

To write a system prompt that makes the model respond as a "Senior Materials Scientist in semiconductor devices".

In [10]:
def exercise_1(pipe):

    print("\n--- EXERCISE 1: Role Prompting ---")

    system_prompt = """ Behave as a "Senior Materials Scientist with 20 years of experience" not a general assistant or 2nd class graduate student. Additionally,
    - Be quantitave and use technical terminology,
    - Always provide numerical values with units
    - Use the temperature when discussing temperature-dependent properties
    - Be concise (3 sentences max).
    """

    user_question = "How does doping affect silicon's electrical properties?"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_question}
    ]

    response = generate(pipe, messages, max_new_tokens=150)
    print(f"Response: {response}")



In [11]:
pipe = load_pipeline()
    
exercise_1(pipe)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- EXERCISE 1: Role Prompting ---
Response:  Doping silicon with impurities such as phosphorus (n-type) or boron (p-type) introduces additional charge carriers, significantly increasing its electrical conductivity. For instance, doping silicon with 1 part per million (ppm) of phosphorus can increase its conductivity by approximately 10^3 S/m. At room temperature (25°C), this doping can shift the Fermi level closer to the conduction band in n-type silicon, enhancing electron mobility and thus the material's overall conductivity.


In [18]:
def exercise_2(pipe):
    """
    Few-Shot Learning: Build a few-shot prompt that classifies materials into their crystal structure category: FCC, BCC, HCP, or Diamond Cubic.

    - Titanium: HCP (at room temperature)
    - Molybdenum: BCC
    - Lead: FCC
    - Germanium: Diamond Cubic
    """
    print("\n--- EXERCISE 2: Few-Shot Crystal Structure Classification ---")

    test_materials = ["Titanium", "Molybdenum", "Lead", "Germanium"]

    for material in test_materials:

        prompt = f"""If
        Material: Aluminum -> Crystal Structure: FCC (at what temperature),
        Material: {material} -> Crystal Structure:
        """

        messages = [{"role": "user", "content": prompt}]

        response = generate(pipe, messages, max_new_tokens=20)
        print(f"{material}: {response.strip()}")

In [19]:
exercise_2(pipe)

Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- EXERCISE 2: Few-Shot Crystal Structure Classification ---


Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Titanium: For aluminum, the face-centered cubic (FCC) crystal structure is


Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Molybdenum: For aluminum, the face-centered cubic (FCC) crystal structure is


Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Lead: For aluminum, the face-centered cubic (FCC) crystal structure is
Germanium: For aluminum, the face-centered cubic (FCC) crystal structure is


In [ ]:
def exercise_3(pipe):
    """
    Chain-of-Thought Reasoning

    "A copper wire has a diameter of 2mm and length of 100m. Calculate its electrical resistance at 25°C, given that copper's resistivity at 20°C is 1.68e-8 Ω·m and its temperature coefficient is 3.9e-3 /°C."

    Answer: ~0.538 Ω
    """
    print("\n--- EXERCISE 3: Chain-of-Thought for Materials Calculation ---")

    problem = (
        "A copper wire has a diameter of 2mm and length of 100m. Calculate its electrical resistance at 25°C, given that copper's resistivity at 20°C is 1.68e-8 Ω·m and its temperature coefficient is 3.9e-3 /°C."
    )

    messages = [
        {
            "role": "system",
            "content": "Behave like a first class metallurgical and materials engineer!"
        },
        {
            "role": "user",
            "content": "Follow this approach, Identify the formula, Calculate the cross-sectional area, Adjust resistivity for temperature, Calculate resistance, State answer with units."
        }
    ]

    response = generate(pipe, messages, max_new_tokens=1000)
    print(response)



In [23]:
exercise_3(pipe)

Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- EXERCISE 3: Chain-of-Thought for Materials Calculation ---
 To solve this problem, we need to follow the steps outlined in the instruction. However, since no specific material or dimensions are provided, I'll use a hypothetical example to demonstrate the process.

Let's assume we have a copper wire with a length (L) of 2 meters and a diameter (d) of 0.5 mm. The resistivity (ρ) of copper at 20°C is approximately 1.68 x 10^-8 Ω·m. We'll calculate the resistance (R) of this wire at 20°C and then adjust it for a temperature of 100°C.

Step 1: Identify the formula
The formula for resistance (R) of a wire is given by:

R = ρ * (L / A)

where ρ is the resistivity, L is the length, and A is the cross-sectional area.

Step 2: Calculate the cross-sectional area
The cross-sectional area (A) of a wire with a circular cross-section is given by:

A = π * (d / 2)^2

where d is the diameter of the wire.

For our example, the diameter (d) is 0.5 mm, which is equal to 0.0005 m. So, the cross-sectio

In [24]:
def exercise_4(pipe):
    """
    Prompt Templates

    Create a PromptTemplate class (or use a dict-based approach) for summarizing materials science paper abstracts.
    """
    print("\n--- EXERCISE 4: Prompt Template for Literature Summarization ---")

    # Sample abstracts for testing
    abstract_1 = """
    We report the synthesis of a novel high-entropy alloy (HEA) composed of CrMnFeCoNi with additions of 2 at.% Cu. The alloy was prepared by arc melting and characterized by XRD, TEM, and tensile testing. The material exhibits a single-phase FCC structure with yield strength of 450 MPa at room temperature, superior to the base Cantor alloy. Hardness measurements confirm a 15% improvement over the five-component system.
    """

    def summarize_abstract(pipe, abstract: str, focus_area: str, audience: str) -> str:
        prompt = f"Summarize the following abstract, focusing on {focus_area} and tailored for a {audience}: {abstract}, Include main finding, key materials, and relevance to {focus_area}."
        messages = [{"role": "user", "content": prompt}]
        return generate(pipe, messages, max_new_tokens=200)

    summary = summarize_abstract(
        pipe,
        abstract=abstract_1,
        focus_area="mechanical properties",
        audience="graduate student"
    )
    print(f"Summary:\n{summary}")



In [25]:
exercise_4(pipe)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- EXERCISE 4: Prompt Template for Literature Summarization ---
Summary:
 The study introduces a new high-entropy alloy (HEA) made of CrMnFeCoNi with 2 at.% Cu, synthesized through arc melting. Characterized by XRD, TEM, and tensile testing, this HEA displays a single-phase FCC structure and a yield strength of 450 MPa at room temperature, which surpasses the base Cantor alloy. Hardness tests reveal a 15% enhancement compared to the original five-component system. This finding is significant for graduate students interested in the mechanical properties of HEAs, as it demonstrates the potential for improved strength and hardness in these materials.


In [26]:
def exercise_5(pipe):
    """
    Build a prompt that extracts material properties from unstructured text and returns them as valid, parseable JSON.
    """
    print("\n--- EXERCISE 5: Structured JSON Extractor ---")

    input_text = """
    Inconel 718 is a nickel-based superalloy widely used in jet engines. It maintains its strength up to 700°C and has a tensile strength of about 1240 MPa. The alloy has a density of 8.19 g/cm³ and melts between 1260-1336°C. It's primarily composed of Ni (50-55%), Cr (17-21%), and Fe (balance), with additions of Nb, Mo, Ti, and Al.
    """

    # TODO: Build a prompt that reliably extracts JSON from the above text
    # Hints:
    # - Specify "respond with ONLY valid JSON, no markdown, no explanation"
    # - Provide the schema explicitly in the prompt
    # - Consider using a few-shot example with a simpler material first
    messages = [
        {
            "role": "system",
            "content": """using the following extracting way and outputs JSON as per the following schema {
                                    "name": string,
                                    "material_class": string,
                                    "composition": {element: percentage},
                                    "max_service_temp_C": number,
                                    "tensile_strength_MPa": number,
                                    "density_g_cm3": number,
                                    "melting_range_C": [low, high],
                                    "primary_applications": [string]
                                }"""
        },
        {
            "role": "user",
            "content": f"Build your extraction prompt for:\n{input_text}"
        }
    ]

    response = generate(pipe, messages, max_new_tokens=400)
    print(f"Raw response: {response}")
    try:
        import re
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            print(f"✓ JSON parsed successfully!")
            print(json.dumps(parsed, indent=2))
        else:
            print("✗ No JSON found in response")
    except json.JSONDecodeError as e:
        print(f"✗ JSON parsing failed: {e}")


In [27]:
exercise_5(pipe)

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- EXERCISE 5: Structured JSON Extractor ---
Raw response:  ```json

{

  "name": "Inconel 718",

  "material_class": "nickel-based superalloy",

  "composition": {

    "Ni": "50-55%",

    "Cr": "17-21%",

    "Fe": "balance",

    "Nb": "0.2-0.4%",

    "Mo": "0.3-0.7%",

    "Ti": "0.1-0.3%",

    "Al": "0.1-0.3%"

  },

  "max_service_temp_C": 700,

  "tensile_strength_MPa": 1240,

  "density_g_cm3": 8.19,

  "melting_range_C": [1260, 1336],

  "primary_applications": ["jet engines"]

}

```
✓ JSON parsed successfully!
{
  "name": "Inconel 718",
  "material_class": "nickel-based superalloy",
  "composition": {
    "Ni": "50-55%",
    "Cr": "17-21%",
    "Fe": "balance",
    "Nb": "0.2-0.4%",
    "Mo": "0.3-0.7%",
    "Ti": "0.1-0.3%",
    "Al": "0.1-0.3%"
  },
  "max_service_temp_C": 700,
  "tensile_strength_MPa": 1240,
  "density_g_cm3": 8.19,
  "melting_range_C": [
    1260,
    1336
  ],
  "primary_applications": [
    "jet engines"
  ]
}


### Embeddings and Vector representations

In [ ]:
### Sample abstracts

MATERIALS_CORPUS = [
    {
        "id": 0,
        "title": "Silicon photovoltaics: efficiency limits",
        "text": "Silicon solar cells dominate the photovoltaic market. The theoretical "
                "Shockley-Queisser limit for silicon is 29.4% due to its 1.12 eV band gap. "
                "Current commercial cells achieve 22-26% efficiency."
    },
    {
        "id": 1,
        "title": "GaAs high-efficiency solar cells",
        "text": "Gallium arsenide single-junction cells achieve 28.9% efficiency under AM1.5G. "
                "The direct band gap of 1.42 eV enables efficient photon absorption. "
                "GaAs is preferred for space applications due to radiation hardness."
    },
    {
        "id": 2,
        "title": "Lithium-ion battery cathode materials",
        "text": "LiCoO2, LiFePO4, and NMC are common cathode materials for Li-ion batteries. "
                "Energy density, cycle life, and thermal stability vary significantly. "
                "NMC811 provides high energy density but has thermal runaway concerns."
    },
    {
        "id": 3,
        "title": "Solid-state electrolytes for batteries",
        "text": "Solid-state batteries replace liquid electrolytes with ceramic or polymer conductors. "
                "Li7La3Zr2O12 (LLZO) garnet shows ionic conductivity of 1 mS/cm. "
                "These materials eliminate flammability risks but have high interface resistance."
    },
    {
        "id": 4,
        "title": "High-entropy alloys: mechanical properties",
        "text": "High-entropy alloys (HEAs) contain 5+ principal elements in near-equimolar ratios. "
                "The Cantor alloy CrMnFeCoNi exhibits exceptional fracture toughness at cryogenic temperatures. "
                "Solid solution strengthening and sluggish diffusion contribute to thermal stability."
    },
    {
        "id": 5,
        "title": "Shape memory alloys: Nitinol",
        "text": "Nitinol (NiTi) exhibits shape memory effect and superelasticity due to martensitic transformation. "
                "Transformation temperature can be tuned from -50°C to 100°C by composition. "
                "Applications include medical stents, actuators, and aerospace deployables."
    },
    {
        "id": 6,
        "title": "Perovskite solar cells: recent advances",
        "text": "Methylammonium lead iodide (MAPbI3) perovskite cells surpass 25% efficiency. "
                "Lead-free alternatives like tin and bismuth perovskites are under investigation. "
                "Stability under moisture and heat remains the primary challenge."
    },
    {
        "id": 7,
        "title": "Graphene synthesis and properties",
        "text": "Graphene is a single-layer carbon lattice with exceptional electrical conductivity (10^8 S/m). "
                "CVD synthesis on copper foil enables large-area graphene films. "
                "Applications include flexible electronics, sensors, and composite reinforcement."
    },
    {
        "id": 8,
        "title": "Superalloys for turbine blades",
        "text": "Nickel-based superalloys like Inconel 718 maintain strength above 700°C. "
                "Single-crystal casting eliminates grain boundary failure modes. "
                "Thermal barrier coatings of yttria-stabilized zirconia extend service life."
    },
    {
        "id": 9,
        "title": "Transparent conducting oxides",
        "text": "Indium tin oxide (ITO) is the dominant transparent conductor for displays and solar cells. "
                "Sheet resistance of 10-100 Ω/sq with >85% optical transmittance. "
                "Alternatives include aluminum-doped zinc oxide (AZO) and carbon nanotube films."
    },
]

In [33]:
### Generating embeddings for above data corpus using sentence-transformers

def demo_generating_embeddings():
    """
    Model choices:
    - all-MiniLM-L6-v2 (22MB)
    - all-mpnet-base-v2 (420MB)
    - scibert-scivocab-uncased (Science-specific vocabulary)
    """
    print("=" * 60)
    print("SECTION 1: Generating Embeddings")
    print("=" * 60)

    print("\nLoading sentence-transformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2') # a fast, lightweight embedding model
    print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

    # Some sample texts
    texts = [
        "Silicon is a semiconductor with band gap 1.12 eV",
        "GaAs has a direct band gap suitable for solar cells",
        "Copper is an excellent electrical conductor",
        "Lithium-ion batteries power electric vehicles",
    ]

    print("\nGenerating embeddings...")
    embeddings = model.encode(texts, show_progress_bar=True)

    print(f"\nEmbedding matrix shape: {embeddings.shape}")
    print(f"Each text -> vector of shape ({embeddings.shape[1]},)")
    print(f"Data type: {embeddings.dtype}")
    print(f"\nFirst 8 dimensions of embedding for '{texts[0][:30]}...':")
    print(f"  {embeddings[0, :8]}")
    print(f"\nL2 norm of embedding:")
    print(f"  {np.linalg.norm(embeddings[0]):.4f}")

    return model, embeddings, texts

In [34]:
model, embeddings, texts = demo_generating_embeddings()

SECTION 1: Generating Embeddings

Loading sentence-transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Embedding dimension: 384

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (4, 384)
Each text -> vector of shape (384,)
Data type: float32

First 8 dimensions of embedding for 'Silicon is a semiconductor wit...':
  [ 0.00287539  0.10464123 -0.04042891  0.01806891 -0.01655888 -0.0687622
  0.06007306  0.07175449]

L2 norm of embedding:
  1.0000


In [42]:
### Computing cosine similarity between embeddings to measure similarity between materials science sentences

def demo_cosine_similarity(model):
    print("\n" + "=" * 60)
    print("SECTION 2: Cosine Similarity")
    print("=" * 60)

    # Materials science sentences with varying similarity
    sentence_pairs = [
        ("Silicon solar cell efficiency", "Photovoltaic power conversion"),  
        ("GaAs band gap 1.42 eV", "Silicon band gap 1.12 eV"),               
        ("NiTi shape memory alloy", "Li-ion battery cathode"),
        ("Copper electrical conductivity", "Gold electrical resistivity"),
    ]

    print("\nPairwise Cosine Similarity:")
    print("-" * 65)
    print(f"{'Text A':<35} | {'Text B':<25} | Sim")
    print("-" * 65)

    for text_a, text_b in sentence_pairs:
        emb_a = model.encode(text_a)
        emb_b = model.encode(text_b)
        sim = util.cos_sim(emb_a, emb_b).item()
        print(f"{text_a[:33]:<35} | {text_b[:23]:<25} | {sim:.4f}")

    # Demonstrate asymmetry vs. directional similarity
    print("\n--- Nearest Neighbor in Embedding Space ---")
    materials = [
        "iron oxide", "hematite", "magnetite",
        "silicon carbide", "silicon nitride",
        "polyethylene", "polypropylene",
    ]

    embeddings = model.encode(materials)
    sim_matrix = util.cos_sim(embeddings, embeddings).numpy()

    print("\nSimilarity matrix:")
    print(" " * 18 + "  ".join([f"{m[:12]:<12}" for m in materials]))
    for i, mat in enumerate(materials):
        row = "  ".join([f"{sim_matrix[i,j]:.3f}   " for j in range(len(materials))])
        print(f"{mat[:16]:<18} {row}")

In [43]:
demo_cosine_similarity(model)


SECTION 2: Cosine Similarity

Pairwise Cosine Similarity:
-----------------------------------------------------------------
Text A                              | Text B                    | Sim
-----------------------------------------------------------------
Silicon solar cell efficiency       | Photovoltaic power conv   | 0.5211
GaAs band gap 1.42 eV               | Silicon band gap 1.12 e   | 0.7390
NiTi shape memory alloy             | Li-ion battery cathode    | 0.1439
Copper electrical conductivity      | Gold electrical resisti   | 0.5461

--- Nearest Neighbor in Embedding Space ---

Similarity matrix:
                  iron oxide    hematite      magnetite     silicon carb  silicon nitr  polyethylene  polypropylen
iron oxide         1.000     0.487     0.540     0.316     0.296     0.207     0.258   
hematite           0.487     1.000     0.363     0.251     0.225     0.212     0.236   
magnetite          0.540     0.363     1.000     0.358     0.326     0.320     0.301   
sil

In [45]:
### FAISS (Facebook AI Similarity Search) vector index.

def demo_faiss_index(model):
    """
    Index types:
    - IndexFlatL2: Exact search, O(n) per query (use for <100K vectors)
    - IndexFlatIP: Exact inner product (use with normalized vectors for cosine)
    - IndexIVFFlat: Approximate, 10-100x faster (use for >100K vectors)
    - IndexHNSWFlat: Graph-based, very fast approximate (use for production)
    """
    print("\n" + "=" * 60)
    print("SECTION 3: Building a FAISS Vector Index")
    print("=" * 60)

    corpus_texts = [doc["text"] for doc in MATERIALS_CORPUS]
    corpus_titles = [doc["title"] for doc in MATERIALS_CORPUS]

    print(f"\nEncoding {len(corpus_texts)} materials science documents...")
    corpus_embeddings = model.encode(corpus_texts, show_progress_bar=True)

    faiss.normalize_L2(corpus_embeddings)

    dimension = corpus_embeddings.shape[1]
    print(f"Embedding dimension: {dimension}")

    index = faiss.IndexFlatIP(dimension)
    index.add(corpus_embeddings.astype(np.float32))
    print(f"Index built: {index.ntotal} vectors stored")

    return index, corpus_embeddings, corpus_titles, model

In [46]:
index, corpus_embeddings, corpus_titles, model = demo_faiss_index(model)


SECTION 3: Building a FAISS Vector Index

Encoding 10 materials science documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimension: 384
Index built: 10 vectors stored


In [47]:
### Perform semantic search: find the most relevant documents for a query.

def demo_semantic_search(index, corpus_titles, model):
   
    def search(query: str, k: int = 3) -> List[Tuple[str, float]]:
        """Return top-k documents for a query."""
        query_embedding = model.encode([query]).astype(np.float32)
        faiss.normalize_L2(query_embedding)

        scores, indices = index.search(query_embedding, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            results.append((corpus_titles[idx], float(score)))
        return results

    queries = [
        "How efficient are solar cells?",
        "What materials are used in electric vehicle batteries?",
        "High temperature resistant alloys for aerospace",
        "Electronic conductors and transparent films",
    ]

    print("\nSemantic Search Results:")
    print("=" * 60)
    for query in queries:
        results = search(query, k=3)
        print(f"\nQuery: '{query}'")
        for rank, (title, score) in enumerate(results, 1):
            print(f"  {rank}. [{score:.4f}] {title}")

In [53]:
### Visualize the embedding space using PCA and t-SNE.

def demo_embedding_visualization(model):
    # Create a more diverse set of materials for visualization
    materials_by_category = {
        "Semiconductors": ["silicon", "germanium", "GaAs", "InP", "GaN", "CdTe"],
        "Metals": ["copper", "aluminum", "iron", "titanium", "gold", "nickel"],
        "Battery": ["LiCoO2", "LiFePO4", "graphite anode", "solid electrolyte", "NMC cathode"],
        "Ceramics": ["alumina", "zirconia", "silicon carbide", "silicon nitride", "boron nitride"],
    }

    labels = []
    texts = []
    colors = []
    color_map = {"Semiconductors": "#FF6B6B", "Metals": "#4ECDC4", "Battery": "#45B7D1", "Ceramics": "#96CEB4"}

    for category, items in materials_by_category.items():
        for item in items:
            texts.append(item)
            labels.append(f"{category}: {item}")
            colors.append(color_map[category])

    print(f"Embedding {len(texts)} materials for visualization...")
    embeddings = model.encode(texts)

    # PCA projection to 2D
    pca = PCA(n_components=2, random_state=42)
    coords_pca = pca.fit_transform(embeddings)

    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

    # t-SNE projection (better for non-linear structure)
    tsne = TSNE(n_components=2, perplexity=5, random_state=42, max_iter=1000)
    coords_tsne = tsne.fit_transform(embeddings)

    # Plot both
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, coords, title in [
        (axes[0], coords_pca, f"PCA (var explained: {pca.explained_variance_ratio_.sum():.0%})"),
        (axes[1], coords_tsne, "t-SNE")
    ]:
        for category, color in color_map.items():
            mask = [c == color for c in colors]
            cat_coords = coords[mask]
            ax.scatter(cat_coords[:, 0], cat_coords[:, 1],
                       c=color, label=category, s=100, alpha=0.8)
            # Add text labels
            idxs = [i for i, c in enumerate(colors) if c == color]
            for idx in idxs:
                ax.annotate(texts[idx], coords[idx], fontsize=7,
                            xytext=(3, 3), textcoords='offset points')
        ax.set_title(title)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle("Materials Embedding Space Visualization", fontsize=14)
    plt.tight_layout()
    plt.savefig("embedding_visualization.png", dpi=100, bbox_inches='tight')

In [54]:
demo_semantic_search(index, corpus_titles, model)
demo_embedding_visualization(model)


Semantic Search Results:

Query: 'How efficient are solar cells?'
  1. [0.6757] Silicon photovoltaics: efficiency limits
  2. [0.4043] Transparent conducting oxides
  3. [0.3933] GaAs high-efficiency solar cells

Query: 'What materials are used in electric vehicle batteries?'
  1. [0.5458] Solid-state electrolytes for batteries
  2. [0.5052] Lithium-ion battery cathode materials
  3. [0.3211] Transparent conducting oxides

Query: 'High temperature resistant alloys for aerospace'
  1. [0.5124] High-entropy alloys: mechanical properties
  2. [0.4397] Superalloys for turbine blades
  3. [0.3082] Shape memory alloys: Nitinol

Query: 'Electronic conductors and transparent films'
  1. [0.5537] Transparent conducting oxides
  2. [0.4200] Graphene synthesis and properties
  3. [0.3187] GaAs high-efficiency solar cells
Embedding 22 materials for visualization...
PCA explained variance: 21.9%


### Vector and Embeddings Representation Exercise

In [55]:
MODEL = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def exercise_1():
    """
    Exercise 1: K-Means Clustering of Materials

    Given these 15 materials, use K-Means on their embeddings to
    automatically discover 3 clusters. Report which materials
    fall into each cluster and whether the clusters make chemical sense.

    Expected clusters (roughly):
    - Cluster A: Semiconductors/Photovoltaics
    - Cluster B: Structural metals/alloys
    - Cluster C: Battery/electrochemical materials
    """
    print("\n--- EXERCISE 1: Semantic Clustering ---")

    materials = [
        "silicon solar cell", "GaAs photovoltaic", "perovskite cell",
        "CdTe thin film solar", "organic photovoltaic",
        "titanium alloy", "steel structural", "aluminum aerospace",
        "nickel superalloy", "magnesium lightweight",
        "LiFePO4 battery", "NMC cathode", "solid electrolyte",
        "graphite anode", "lithium metal battery"
    ]

    # TODO:
    # 1. Generate embeddings for all materials
    # 2. Apply K-Means with k=3 (use sklearn.cluster.KMeans)
    # 3. Print which materials are in each cluster
    # 4. Comment: do the clusters make chemical/application sense?

    model = get_model()
    # embeddings = model.encode(materials)
    # from sklearn.cluster import KMeans
    # kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    # labels = kmeans.fit_predict(embeddings)
    # ... group and print

    print("(TODO: implement clustering)")
